In [ ]:
!pip install update scipy==1.7

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import torch
from scipy.stats import rankdata
from sklearn.metrics import roc_auc_score
from scipy.optimize import minimize
from scipy.special import expit as sigmoid
from pathlib import Path
# import seaborn as sns
from scipy.optimize import minimize, fsolve
import os

In [ ]:
import scipy
scipy.__version__

In [ ]:
class Config:

    #frequently changed 

    model_version = "124th_V2SDCBAM_PL_6ep_2em3lr_32ch_vf+gn+sc01+tm+ts"#
    


    output_dir = "G2Net-Model/"
    kaggle_json_path = 'kaggle/kaggle.json'
    train_folds = [0]


# no need to change below
Config.model_output_folder = Config.output_dir + Config.model_version + "/"
if not os.path.exists(Config.output_dir):
    os.mkdir(Config.output_dir)
if not os.path.exists(Config.model_output_folder):
    os.mkdir(Config.model_output_folder)

    

display(Config.model_output_folder)

In [ ]:
oof_all = pd.read_csv(f'{Config.model_output_folder}/oof_all.csv')

In [ ]:
oof_all

In [ ]:
oof_all.groupby("target")["preds"].hist(bins=100, alpha=0.5, range=[0, 1],log=True)

In [ ]:
# plt.figure(figsize=(8,6))
# plt.hist(oof, bins=100, alpha=0.5, range=[0, 1])
# plt.suptitle(f"Unit: {pre_method}")
# plt.legend(loc='upper right');

In [ ]:
!free -g

In [ ]:
oof_df = oof_all

In [ ]:
def score_oof(oof, full=False, col=None, get_list=False):
    if col is None:
        col = "prediction"
    if full:
        return round(roc_auc_score(oof["target"], oof[col]),6)
    else:
        scores = [roc_auc_score(df["target"], df[col]) for _, df in oof.groupby("fold")]
        if get_list:
            return scores
        return round(np.mean(scores),6)
    
def score_oof_fast(oof, full=False, col=None, get_list=False):
    if col is None:
        col = "prediction"
    if full:
        return round(fast_auc(oof["target"], oof[col]),6)
    else:
        scores = [fast_auc(df["target"], df[col]) for _, df in oof.groupby("fold")]
        if get_list:
            return scores
        return round(np.mean(scores),6)

from numba import jit

@jit
def fast_auc(y_true, y_prob):
    y_true = np.asarray(y_true)
    y_true = y_true[np.argsort(y_prob)]
    nfalse = 0
    auc = 0
    n = len(y_true)
    for i in range(n):
        y_i = y_true[i]
        nfalse += (1 - y_i)
        auc += y_i * nfalse
    auc /= (nfalse * (n - nfalse))
    return auc

In [ ]:
%%time
sci_opt_preds = oof_df.copy() 
model_cols = oof_df.columns[4:-1]
X = sci_opt_preds[model_cols].values
print(model_cols, sci_opt_preds.shape)
def get_score(weights, oof=sci_opt_preds):
    weights = np.array(weights)
    sci_opt_preds["prediction"] = (weights * X).sum(axis=1)
    return -score_oof_fast(sci_opt_preds)

tol = 1e-10
init_guess = [1 / len(model_cols)] * len(model_cols)
bnds = [(0, 1) for _ in range(len(model_cols))]
print('Inital Blend OOF: {:.6f}'.format(-get_score(init_guess)))
res_scipy = minimize(fun = get_score, 
                    x0 = init_guess, 
                    method = 'Nelder-Mead', 
                    bounds = bnds, 
                    tol = tol)
print('Optimised Blend OOF: {:.6f}'.format(-res_scipy.fun))
sci_cv_score = -res_scipy.fun
print('-' * 70)
for n, key in enumerate(model_cols):
    print(f'{key:40s} Optimised Weights:', res_scipy.x[n])
ws = res_scipy.x 
ws = ws / np.sum(ws) # The nelder-mead method requires normalization because the sum of the weights does not equal 1.
pd.DataFrame(dict(zip(model_cols,ws)),index=["weight"]).T.plot(kind="barh",figsize=(10,16))

In [ ]:
np.sum(ws) 

In [ ]:
for n, key in enumerate(model_cols):
    print(f'{key:40s} Optimised Weights:', ws[n])

In [ ]:
oof_weight = {key:ws[n] for n, key in enumerate(model_cols) }

In [ ]:
oof_weight

In [ ]:
oof_all['avg']=0

print(oof_weight)
total_weight = 0
for key,value in oof_weight.items():
    print(key,value)

for col in oof_all.columns:
    if ('tta_' in col or 'preds' in col): # and 'time_shift' not in col and 'timemask' not in col
        print(col)
        total_weight+=oof_weight[col]
        oof_all['avg'] += oof_all[col]*oof_weight[col]
oof_all['avg'] /= total_weight

print("preds_tta_avg:",roc_auc_score(oof_all['target'], oof_all['avg']))

oof_all.to_csv(Config.model_output_folder + "/oof_all.csv", index=False)
oof_all[['id','fold','avg']].rename(columns={'id':'id','fold':'fold','avg':'prediction'}).to_csv(Config.model_output_folder + "/oof_final.csv", index=False)

In [ ]:
oof_all.groupby("fold").apply(lambda df: roc_auc_score(df['target'],df['avg']))

In [ ]:
def id_2_path(file_id: str, train=True) -> str:
    if train:
        return "./output/whiten-train-w0/{}.npy".format(file_id)
    else:
        return "./output/whiten-test-w0/{}.npy".format(file_id)

In [ ]:
train_df = pd.read_csv('training_labels.csv')
test_df = pd.read_csv('sample_submission.csv')
train_df['file_path'] = train_df['id'].apply(lambda x :id_2_path(x))
test_df['file_path'] = test_df['id'].apply(lambda x :id_2_path(x,False))

In [ ]:
test_avg = test_df[['id', 'target']].copy()
test_avg['target'] = 0
# print(test_avg.describe())

total_weight = 0
for fold in Config.train_folds:
#     test_weight = {key+f'_Fold_{fold}':value for key,value in oof_weight.items()}
    test_weight = oof_weight #defaultdict(lambda:1)
    test_df2 = pd.read_csv(Config.model_output_folder + f"/test_Fold_{fold}.csv")
#     print(test_df2.describe())
    for col in test_df2.columns:
        col_weight = col.split('_Fold_')[0]
        if ('tta_' in col or 'preds' in col): 
#             print(col)
#             print(test_weight[col_weight])
            total_weight+=test_weight[col_weight]
            test_avg['target'] += test_df2[col]*test_weight[col_weight]
test_avg['target'] /= total_weight
print(test_avg.describe())
print(test_avg["target"].hist(bins=100))
print(test_avg)
print(total_weight)
test_avg.to_csv(Config.model_output_folder + "/test_avg.csv", index=False)

In [ ]:
test_avg.describe()

In [ ]:
test_avg[['id', 'target']].to_csv("./submission.csv", index=False)

test_avg[['id', 'target']].to_csv(Config.model_output_folder + "/submission.csv", index=False)

!mkdir -p ~/.kaggle/ && cp $Config.kaggle_json_path ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json

In [ ]:
# !kaggle competitions submit -c g2net-gravitational-wave-detection -f ./submission.csv -m $Config.model_version